[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/claude-api-certified/notebooks/day-07-advanced-features.ipynb#scrollTo=ca01da02)

---
# Day 7 · Advanced Claude Features
**certified-journeys / claude-api-certified** · Thinking, vision, PDFs, caching & citations

> **Goal for today:** Unlock Claude's advanced capabilities — extended thinking for hard reasoning, multimodal inputs for vision and PDFs, prompt caching for 80%+ cost reduction, and citations for verifiable answers.

In [ ]:
%pip install -q anthropic requests Pillow

In [ ]:
import os
import base64
import time
import anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass

client = anthropic.Anthropic()

## Step 1 · Extended Thinking

Extended thinking gives Claude a private reasoning scratchpad before answering. Use it for problems that require multi-step analysis, logical deduction, or careful planning.

| Standard mode | Extended thinking |
|---|---|
| Immediate response | Thinks for N tokens first |
| Good for simple tasks | Better for complex reasoning |
| Cheaper | Uses extra output tokens |
| Available on all models | `claude-sonnet-4-5` and above |

In [ ]:
HARD_PUZZLE = """Five people (Alice, Bob, Carol, Dave, Eve) live in five houses in a row.
Clue 1: Alice lives next to Bob.
Clue 2: Carol lives at one of the ends.
Clue 3: Dave is not next to Eve.
Clue 4: Bob lives in house 3.
Clue 5: Eve is between Dave and Carol.
Who lives in house 5?"""

# Without extended thinking
r_standard = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=512,
    messages=[{"role": "user", "content": HARD_PUZZLE}]
)

# With extended thinking (budget_tokens = max reasoning tokens)
r_thinking = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=8000,
    thinking={"type": "enabled", "budget_tokens": 5000},
    messages=[{"role": "user", "content": HARD_PUZZLE}]
)

print("=== STANDARD ===")
print(r_standard.content[0].text[:300])

print("\n=== EXTENDED THINKING ===")
for block in r_thinking.content:
    if block.type == "thinking":
        print(f"[Thinking: {len(block.thinking)} chars]")
        print(f"First 200 chars: {block.thinking[:200]}...")
    elif block.type == "text":
        print(f"\nFinal answer: {block.text}")

**What just happened?**
- `budget_tokens` controls how much thinking Claude can do — higher = better reasoning, more cost.
- **`ThinkingBlock`** contains Claude's reasoning — useful for debugging and auditing complex decisions.
- Extended thinking **cannot be used with tool use** in the same request (as of mid-2025).
- Start with `budget_tokens=2000` and increase only if answers are still wrong.

## Step 2 · Vision — image inputs

Claude can analyze images passed as base64-encoded data or URLs. Supported formats: JPEG, PNG, GIF, WebP.

In [ ]:
import requests
from PIL import Image, ImageDraw
from io import BytesIO

# Create a simple test image with PIL (no external dependencies)
img = Image.new("RGB", (400, 200), color="#FAEDE8")
draw = ImageDraw.Draw(img)
draw.rectangle([20, 20, 380, 180], outline="#D97757", width=4)
draw.text((50, 80), "Claude API  |  Day 7 Vision Test", fill="#993C1D")
draw.text((50, 120), "Revenue: $1.2M  |  Users: 45,000  |  Growth: 23%", fill="#52524E")

buffer = BytesIO()
img.save(buffer, format="PNG")
img_b64 = base64.standard_b64encode(buffer.getvalue()).decode("utf-8")

r = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=256,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": img_b64
                }
            },
            {
                "type": "text",
                "text": "What metrics are shown in this image? List them."
            }
        ]
    }]
)

print("Claude's vision response:")
print(r.content[0].text)

**What just happened?**
- Vision uses a **list of content blocks** for the message — mixing text and image in one message.
- `source.type: 'base64'` for local images; `source.type: 'url'` for public web images.
- **Image tokens count toward your context window** — a 1000×1000 image uses ~1600 tokens.
- Claude can analyze charts, screenshots, diagrams, forms, and handwritten text.

## Step 3 · Prompt caching — 80%+ cost reduction

Prompt caching stores large prompt sections server-side. Repeated requests that hit the cache pay ~10% of normal input token cost.

In [ ]:
# Simulate a large system prompt (repeat to inflate token count)
LARGE_SYSTEM = ("You are an expert in the Anthropic Claude API. " * 200).strip()

# Count tokens to verify it's large enough for caching (minimum 1024)
token_count = client.messages.count_tokens(
    model="claude-haiku-4-5",
    system=[{"type": "text", "text": LARGE_SYSTEM, "cache_control": {"type": "ephemeral"}}],
    messages=[{"role": "user", "content": "Hello"}]
)
print(f"System prompt tokens: {token_count.input_tokens}")

def call_with_caching(user_question: str, use_cache: bool) -> dict:
    system = [
        {
            "type": "text",
            "text": LARGE_SYSTEM,
            **({
                "cache_control": {"type": "ephemeral"}
            } if use_cache else {})
        }
    ]
    start = time.time()
    r = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=64,
        system=system,
        messages=[{"role": "user", "content": user_question}]
    )
    elapsed = time.time() - start
    return {
        "latency": round(elapsed, 2),
        "cache_creation": getattr(r.usage, "cache_creation_input_tokens", 0),
        "cache_read": getattr(r.usage, "cache_read_input_tokens", 0),
        "input_tokens": r.usage.input_tokens,
    }


print("\nFirst call (writes cache):")
r1 = call_with_caching("What is the Messages API?", use_cache=True)
print(r1)

time.sleep(1)  # Brief pause between calls

print("\nSecond call (reads cache):")
r2 = call_with_caching("Explain streaming.", use_cache=True)
print(r2)

if r2["cache_read"] > 0:
    print(f"\nCache hit! Saved {r2['cache_read']} tokens from cache.")

**What just happened?**
- `cache_creation_input_tokens` — tokens written to cache on first call (costs 25% more than normal).
- `cache_read_input_tokens` — tokens read from cache (costs ~10% of normal — the big saving).
- **Cache lives for 5 minutes** (ephemeral). Must re-create after expiry.
- Cache is keyed on exact content — any change to the cached block invalidates it.

## Step 4 · Citations in RAG responses

Claude can reference exact passages from documents you provide, enabling verifiable answers with source attribution.

In [ ]:
# Provide documents as structured source content
r = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": [
            {
                "type": "document",
                "source": {
                    "type": "text",
                    "media_type": "text/plain",
                    "data": "Prompt caching stores large prompt prefixes server-side for up to 5 minutes. Cache hits cost approximately 10% of standard input token prices. The minimum cacheable block size is 1024 tokens. Add cache_control with type 'ephemeral' to enable caching on a content block."
                },
                "title": "Anthropic Docs: Prompt Caching",
                "citations": {"enabled": True}
            },
            {
                "type": "text",
                "text": "Based on the document above, what is the minimum block size for caching and what does it cost?"
            }
        ]
    }]
)

print("Response with citations:")
for block in r.content:
    if block.type == "text":
        print(block.text)
    elif hasattr(block, "citations"):
        print(f"\nCitations: {block.citations}")

**What just happened?**
- `citations: {"enabled": True}` tells Claude to reference exact passages from the document.
- Citations appear as structured metadata alongside the text response.
- **Perfect for RAG** — users can verify that answers come from specific source passages.
- Citations only work with `document` type content blocks, not plain text in messages.

## Step 5 · Combine caching + RAG + citations

Real production systems layer these features together. Here's a pattern that caches the knowledge base system prompt and uses citations.

In [ ]:
KB_CONTEXT = """You are an expert on the Claude API. Answer questions using only the provided documents.
Always cite specific passages. If information is not in the documents, say so explicitly.

" "* 100  # Padding to exceed 1024 token minimum for caching"""

KNOWLEDGE_DOCS = [
    {"title": "Token Counting", "content": "Use client.messages.count_tokens() to estimate token usage before making a real API call. This method is free — it does not generate any output. It accepts the same parameters as messages.create()."},
    {"title": "Streaming", "content": "Enable streaming with client.messages.stream() as a context manager. Use stream.text_stream to iterate over text chunks. Call stream.get_final_message() after the stream closes to access usage statistics."},
]

def cached_rag_with_citations(question: str) -> str:
    """RAG call with cached system prompt and document citations."""
    content = []
    for doc in KNOWLEDGE_DOCS:
        content.append({
            "type": "document",
            "source": {"type": "text", "media_type": "text/plain", "data": doc["content"]},
            "title": doc["title"],
            "citations": {"enabled": True}
        })
    content.append({"type": "text", "text": question})

    r = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=512,
        system=[
            {"type": "text", "text": KB_CONTEXT, "cache_control": {"type": "ephemeral"}}
        ],
        messages=[{"role": "user", "content": content}]
    )
    return r.content[0].text if r.content else ""


answer = cached_rag_with_citations("How do I count tokens and what does it cost?")
print(answer)

**What just happened?**
- **System prompt caching** + **document citations** + **RAG retrieval** work together in a single request.
- The cached system prompt covers the persistent instruction set; documents are dynamic per query.
- This pattern is the foundation of production knowledge assistants with verifiable answers.
- Monitor `cache_read_input_tokens` in production to verify your caching strategy is working.

In [ ]:
# Challenge: Add PDF processing
# The Claude API accepts PDFs as document source blocks.
# Create a function that takes a PDF file path, reads it as base64,
# and sends it to Claude with a question.

def analyze_pdf(pdf_path: str, question: str) -> str:
    """Read a PDF file and ask Claude a question about it."""
    # Your solution here:
    # 1. Read pdf_path as bytes
    # 2. Encode to base64
    # 3. Build a message with source type 'base64' and media_type 'application/pdf'
    # 4. Return Claude's answer
    pass


# Test skeleton (you'd need a real PDF to test fully)
print("analyze_pdf() skeleton ready — implement above and test with a real PDF.")
print("Hint: source = {type: 'base64', media_type: 'application/pdf', data: b64_string}")

---
## Day 7 key concepts recap

| Feature | When to use | Key parameter |
|---|---|---|
| Extended thinking | Complex reasoning, logic puzzles | `thinking.budget_tokens` |
| Vision | Image analysis, chart reading, OCR | `source.type: 'base64'` or `'url'` |
| Prompt caching | Repeated large system prompts | `cache_control: {type: 'ephemeral'}` |
| Citations | Verifiable RAG answers | `citations: {enabled: True}` on documents |

> **Tip:** Prompt caching cuts costs 80%+ on repeated large contexts. Cache anything > 1000 tokens that doesn't change between requests.

---
## What's next
**Day 8** → Model Context Protocol: build MCP servers, clients, and debug with the MCP Inspector.

Mark Day 7 complete in your [tracker](../index.html).